# ATML PA2 — Task 3: GRPO
Use an A100 GPU runtime. Artifacts persist in `MyDrive/PA2`, alongside Tasks 1–2. This notebook runs the prescribed experiments and exports numeric evidence. Write your own report analysis and qualitative judgments. Run cells from top to bottom.


In [ ]:
from google.colab import drive, files
from pathlib import Path
import os, sys, subprocess, shutil, json, signal

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/PA2')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['PA2_ARTIFACT_ROOT'] = str(DRIVE_ROOT)
os.environ['USE_TF'] = '0'
os.environ['PYTHONUNBUFFERED'] = '1'
REPO = Path('/content/ATML')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/WasifAhmed1917/ATML.git', str(REPO)], check=True)
else:
    subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO, check=True)
PA2 = REPO / 'PA2'
os.chdir(PA2)
assert (PA2 / 'task3_grpo/run_all.py').exists(), 'Task 3 code must be published to GitHub first.'
print('Code revision:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip())
print('Persistent artifacts:', DRIVE_ROOT)


## Dependencies and live output
Every subprocess streams tqdm and saves a Drive log. Exit codes are checked: a failed command stops the cell. After a disconnected runtime, rerun setup, dependencies, and validation, then the unfinished stage with `--resume`. Keep the same code revision while resuming partial runs.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)

def run(module, *args, artifact_root=None):
    env = {**os.environ, 'PYTHONUNBUFFERED': '1', 'USE_TF': '0'}
    if artifact_root is not None:
        env['PA2_ARTIFACT_ROOT'] = str(artifact_root)
    log_dir = DRIVE_ROOT / 'logs/task3_grpo'
    log_dir.mkdir(parents=True, exist_ok=True)
    label = module.replace('.', '_') + ('_' + args[args.index('--stage')+1] if '--stage' in args else '')
    command = [sys.executable, '-u', '-m', module, *args]
    print('Running:', ' '.join(command), flush=True)
    with (log_dir / (label + '.log')).open('a') as log:
        process = subprocess.Popen(command, cwd=PA2, env=env, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line); log.flush()
            status = process.wait()
        except BaseException:
            try:
                os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
            process.wait()
            raise
    print('Exit code:', status)
    if status:
        raise subprocess.CalledProcessError(status, command)
run('scripts.check_environment')


## Course assets and objective validation
The downloader uses the pinned asset revision. All continuations start from the supplied GRPO midpoint with a fresh optimizer. Tests check within-prompt advantages, clipping, truncation, both normalizations, actual tiny-Qwen2 optimization, and checkpoint recovery.


In [ ]:
run('scripts.download_assets')
run('task3_grpo.validate_data')
run('unittest', 'discover', '-s', 'tests', '-p', 'test_task2*.py', '-v')


## GPU smoke test
One update with K=4 and a 16-token cap, then two held-out prompts. To exercise FP16 backward, only this isolated smoke test disables completion-level truncation masking. Full experiments enable the prescribed mask. Smoke artifacts live in a separate folder and cannot satisfy full-run completion checks.


In [ ]:
SMOKE_ROOT = DRIVE_ROOT / 'task3_smoke'
run('task3_grpo.continue_train', '--run-name', 'smoke', '--output', 'outputs/task3_grpo/smoke', '--smoke', '--resume', '--skip-completed', artifact_root=SMOKE_ROOT)
run('task3_grpo.evaluate', '--adapter', 'outputs/task3_grpo/smoke', '--name', 'smoke', '--smoke', '--resume', artifact_root=SMOKE_ROOT)


## Standard: 20-update continuation
Also evaluates the supplied GRPO midpoint on the same 200 held-out prompts. Training uses K=4, a 512-token completion cap, and whole-completion masking for truncation. Evaluation uses a 768-token cap, matching Task 2. Updates with no eligible completions skip the optimizer and are explicitly counted.


In [ ]:
run('task3_grpo.run_all', '--stage', 'standard', '--resume')


## Equal-generation group-size study
Cache only: partition each of the 24 prompts' eight completions into consecutive groups of K=2/4/8. All conditions use 192 generations. Two difficulty bins are fixed once by ranking each prompt's full-eight mean learned reward; these are reward-based proxies. No GPU training is required in this cell.


In [ ]:
run('task3_grpo.run_all', '--stage', 'groups', '--resume')


## Matched normalization forks
Eight updates each: canonical GRPO and the supplied Dr.GRPO-style constant denominator. Both start from the same midpoint and first eight training prompts, with identical seeds/settings and a maximum 16,384-token generation budget. Realized tokens can differ due to EOS. Both retain reward-std normalization and the released KL normalization. Length-conditioned policy-logp gradient statistics are saved directly during replay.


In [ ]:
run('task3_grpo.run_all', '--stage', 'normalization', '--resume')


## Export and plots
Checks full-run completion, matched settings and prompt IDs, cache budgets and truncation masks. Exports summaries, trajectories, group diagnostics, and length-conditioned gradients. `qualitative_review.jsonl` contains responses and blank fields for your manual analysis; report prose is not generated.


In [ ]:
run('task3_grpo.run_all', '--stage', 'export', '--resume')


In [ ]:
import pandas as pd
from IPython.display import display, Image
RESULTS = DRIVE_ROOT / 'results/task3_grpo'
for name in ['summary.csv', 'group_sizes/summary.csv', 'length_conditioned_summary.csv']:
    path = RESULTS / name
    if path.exists(): display(pd.read_csv(path))
for path in sorted((RESULTS / 'figures').glob('*.png')):
    display(Image(filename=str(path)))
print(json.dumps(json.loads((RESULTS / 'completion_checks.json').read_text()), indent=2))


## Save results
The results archive is copied to Drive and downloaded. The standard final adapter remains in `MyDrive/PA2/outputs/task3_grpo/standard` for Task 4. Per-update rollout tensors remain in Drive and are not included in the smaller results-only archive.


In [ ]:
archive = shutil.make_archive('/content/ATML_PA2_T3_results', 'zip', root_dir=DRIVE_ROOT / 'results', base_dir='task3_grpo')
shutil.copy2(archive, DRIVE_ROOT / Path(archive).name)
files.download(archive)
